Name : Naveen Gupta

Class : CSE,4th Year

# AI Agents — Exercise Set 2

**Topic:** ReAct Agents, Custom Tools, Agent Debugging, Memory (LangChain + Groq)

**Instructions:**
* All problems here are NEW (different from Set 1).
* Part A = trace & predict (no coding), Part B = fix broken code, Part C = build, Part D = mini-project.
* Run the setup cells first.

---

## ⚙️ Setup (run first)

In [ ]:
!pip install -q langchain langchain-groq langchain-community

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("GROQ API KEY: ")

GROQ API KEY: ··········


In [ ]:
from langchain_groq import ChatGroq
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain.tools import tool
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate
import math

llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)
print("LLM ready:", llm.model_name)

LLM ready: llama-3.3-70b-versatile


---
# Part A — Trace & Predict 🔍 (No coding, brain only)

For each question, **write down what you think the agent will do BEFORE running anything.**

### A1. Predict the Trace

The agent has tools: `calculator`, `get_weather`, `knowledge_base` (from class).

User asks: **"What is a neural network, and what is 2 to the power 10?"**

Predict the full trace:
* How many Action steps? Which tools, in which order?
* What will each Action Input be?

**Your prediction:**

a)

    Number of Action steps: 2
    Tool order:
        knowledge_base
        calculator



b)

```Thought: I need to answer two different questions. I should use the knowledge_base for the definition and the calculator for the mathematical calculation.

Action: knowledge_base
Action Input: neural network

Observation: A neural network is a machine learning model inspired by the human brain. It consists of interconnected layers of neurons that learn patterns from data.

Thought: Now I need to calculate 2 to the power 10.

Action: calculator
Action Input: 2**10

Observation: 1024

Thought: I now have both pieces of information.

Final Answer:
A neural network is a machine learning model made up of interconnected artificial neurons that learn patterns from data. It is widely used for tasks such as image recognition, language processing, and prediction.

2 to the power of 10 is **1024**.
```

### A2. Spot the Ambiguity

User asks: **"Tell me about React."**

The class `knowledge_base` has an entry for `"react"` (the ReAct prompting strategy). But the user might mean React.js (the JavaScript library)!

1. Which answer will our agent give, and why?
2. How would you rewrite the tool's docstring OR the kb entry to reduce this confusion?

**Your answer:**

1. The agent will most likely return the ReAct prompting strategy because it simply searches the knowledge base for the matching keyword "react". It has no way of knowing that the user might be referring to React.js unless the knowledge base or tool description distinguishes between the two meanings.


2. Improve the tool docstring:
Use this tool only for AI and machine learning concepts stored in the course knowledge base. It does not contain general programming or web development topics.

OR improve the knowledge base entry:

Instead of

    react → ReAct prompting strategy

use

    react_prompting → ReAct prompting strategy used in AI agents


This makes the meaning much clearer.

### A3. The Format Rules

Look at the ReAct prompt from class. It says: *"Action must be exactly one of the tool names listed."*

1. What happens if the LLM writes `Action: Calculator` (capital C) when the tool is named `calculator`?
2. Which `AgentExecutor` parameter helps the agent recover from this kind of mistake?

**Answer:**

### 1)
 Since the action name must exactly match one of the registered tool names, "Calculator" will not match the tool named "calculator".

The agent will fail to execute the tool and typically produce an error such as:

    Invalid tool: Calculator





### 2)

The parameter is:

handle_parsing_errors=True

When enabled, the AgentExecutor catches formatting or parsing errors and allows the LLM to try again instead of immediately failing. This improves the agent's ability to recover from mistakes such as incorrect action formatting or invalid outputs.

### B1. Broken Tool (2 bugs)

Hint: one bug prevents the agent from ever choosing this tool; the other crashes the lookup.

In [ ]:
# # FIX THIS CELL — 2 bugs

# :@tool
# def pincode_lookup(city: str) -> str:
#     pin_db = {
#         "delhi": "110001",
#         "mumbai": "400001",
#         "sonipat": "131001",
#         "bengaluru": "560001",
#     }
#     return pin_db[city]

# # Test after fixing:
# print(pincode_lookup.invoke("Delhi"))       # should work (capital D!)
# print(pincode_lookup.invoke("Chennai"))     # should NOT crash








from langchain.tools import tool

@tool

# BUG: Added a docstring so the agent knows when to use this tool.
def pincode_lookup(city: str) -> str:
    """
    Look up the postal (PIN) code for a given Indian city.
    """
    pin_db = {
        "delhi": "110001",
        "mumbai": "400001",
        "sonipat": "131001",
        "bengaluru": "560001",
    }

    # BUG: Convert input to lowercase and use .get() to avoid KeyError.
    return pin_db.get(city.lower(), f"Sorry, no PIN code found for '{city}'.")

# Test after fixing:
print(pincode_lookup.invoke("Delhi"))      # 110001
print(pincode_lookup.invoke("Chennai"))    # Sorry, no PIN code found for 'Chennai'.

110001
Sorry, no PIN code found for 'Chennai'.


There are 2 bugs:

1. No docstring → LangChain tools need a description/docstring so the agent knows when to use the tool.

2. Dictionary lookup can fail → pin_db[city] raises a KeyError for unknown cities and doesn't handle different letter cases (e.g., "Delhi" vs "delhi").

### B2. Broken Agent Setup (3 bugs)

Hint: check the prompt variables, the executor arguments, and what's missing to stop runaway loops.

In [ ]:
# # FIX THIS CELL — 3 bugs

# broken_prompt = PromptTemplate.from_template("""You are a helpful AI assistant.

# You have access to these tools:
# {tools}

# Question: {input}
# Thought:""")

# broken_agent = create_react_agent(llm=llm, tools=[pincode_lookup], prompt=broken_prompt)

# broken_executor = AgentExecutor(
#     agent=broken_agent,
#     verbose=True
# )

# # Test after fixing:
# # r = broken_executor.invoke({"input": "What is the pincode of Sonipat?"})

# from langchain.prompts import PromptTemplate
# from langchain.agents import create_react_agent, AgentExecutor

# BUG: Added the required {tool_names} and {agent_scratchpad} variables.

broken_prompt = PromptTemplate.from_template("""
You are a helpful AI assistant.

You have access to these tools:
{tools}

Use only these tool names:
{tool_names}

Use the following format:

Question: the input question
Thought: think about what to do
Action: one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (repeat Thought/Action/Observation as needed)
Thought: I now know the final answer
Final Answer: the answer

Question: {input}
Thought: {agent_scratchpad}
""")

broken_agent = create_react_agent(
    llm=llm,
    tools=[pincode_lookup],
    prompt=broken_prompt
)

broken_executor = AgentExecutor(
    agent=broken_agent,
    tools=[pincode_lookup],      # Optional in newer versions, commonly included
    verbose=True,
    # BUG: Added max_iterations to prevent infinite agent loops.
    max_iterations=5
)

# Test
r = broken_executor.invoke({"input": "What is the pincode of Sonipat?"})
print(r["output"])



> Entering new AgentExecutor chain...
Thought: To find the pincode of Sonipat, I need to use the pincode_lookup function, which takes a city name as input and returns the corresponding postal code.

Action: pincode_lookup
Action Input: Sonipat131001Thought: I now know the final answer
Final Answer: 131001

> Finished chain.
131001


**Write your 5 bug explanations here:**

B1 bug 1: Added a docstring to pincode_lookup so the agent knows when to use the tool.

B1 bug 2: Replaced pin_db[city] with pin_db.get(city.lower(), ...) so the lookup is case-insensitive and doesn't raise a KeyError for unknown cities.

B2 bug 1: Added the {tool_names} prompt variable because create_react_agent requires it to tell the LLM which tool names are valid.

B2 bug 2: Added the {agent_scratchpad} prompt variable so the agent can keep track of previous Thought/Action/Observation steps during reasoning.

B2 bug 3: Added max_iterations=5 to AgentExecutor to prevent infinite reasoning loops if the agent fails to reach a final answer.

---
# Part C — Build New Tools 🛠️

### C1. Grade Calculator Tool

Build a tool `grade_calculator` that takes marks (out of 100, as a string) and returns the grade:

| Marks | Grade |
|-------|-------|
| 90+   | A     |
| 75–89 | B     |
| 60–74 | C     |
| 40–59 | D     |
| < 40  | Fail  |

Handle bad input (e.g. `"abc"` or `"150"`) gracefully with an error message.

In [ ]:
# :@tool
# def grade_calculator(marks: str) -> str:
#     """
#     # TODO: write the description
#     """
#     # TODO: convert to number, validate 0-100, return grade
#     pass

# # Tests:
# print(grade_calculator.invoke("92"))    # Grade: A
# print(grade_calculator.invoke("55"))    # Grade: D
# print(grade_calculator.invoke("150"))   # Error message
# print(grade_calculator.invoke("abc"))   # Error message






from langchain.tools import tool

@tool
def grade_calculator(marks: str) -> str:
    """
    Calculate the grade for a student's marks (0-100).
    Input should be a number as a string.
    Returns the grade or an appropriate error message for invalid input.
    """

    marks = marks.strip().strip("'\"")
    try:
        marks = float(marks)

        if marks < 0 or marks > 100:
            return "Error: Marks must be between 0 and 100."

        if marks >= 90:
            return "Grade: A"
        elif marks >= 75:
            return "Grade: B"
        elif marks >= 60:
            return "Grade: C"
        elif marks >= 40:
            return "Grade: D"
        else:
            return "Grade: Fail"

    except ValueError:
        return "Error: Please enter a valid numeric value for marks."

# Tests
print(grade_calculator.invoke("92"))     # Grade: A
print(grade_calculator.invoke("55"))     # Grade: D
print(grade_calculator.invoke("150"))    # Error: Marks must be between 0 and 100.
print(grade_calculator.invoke("abc"))    # Error: Please enter a valid numeric value for marks.

Grade: A
Grade: D
Error: Marks must be between 0 and 100.
Error: Please enter a valid numeric value for marks.


### C2. Unit Converter Tool (with parsing!)

Build `unit_converter` that handles input like `"5 km to miles"` or `"10 kg to pounds"`.

Support at least: km↔miles (1 km = 0.621 mi) and kg↔pounds (1 kg = 2.205 lb).

Hint: `.split()` the input string. This is harder than it looks — the agent sends free-form text!

In [ ]:
# :@tool
# def unit_converter(query: str) -> str:
#     """
#     # TODO: description — tell the agent the EXACT input format,
#     # e.g. '<number> <unit> to <unit>' like '5 km to miles'
#     """
#     # TODO: parse and convert
#     pass

# # Tests:
# print(unit_converter.invoke("5 km to miles"))     # ~3.11 miles
# print(unit_converter.invoke("10 kg to pounds"))   # ~22.05 pounds







from langchain.tools import tool

@tool
def unit_converter(query: str) -> str:
    """
    Convert units using the exact input format:
    '<number> <unit> to <unit>'

    Examples:
    - '5 km to miles'
    - '10 kg to pounds'
    - '3.1 miles to km'
    - '20 pounds to kg'

    Supported conversions:
    - km ↔ miles
    - kg ↔ pounds
    """

    query = query.strip().strip("'\"")
    try:
        parts = query.lower().split()

        # Expected format: <number> <unit> to <unit>
        if len(parts) != 4 or parts[2] != "to":
            return "Error: Use the format '<number> <unit> to <unit>', e.g. '5 km to miles'."

        value = float(parts[0])
        from_unit = parts[1]
        to_unit = parts[3]

        # Distance conversions
        if from_unit == "km" and to_unit == "miles":
            result = value * 0.621
            return f"{value} km = {result:.2f} miles"

        elif from_unit == "miles" and to_unit == "km":
            result = value / 0.621
            return f"{value} miles = {result:.2f} km"

        # Weight conversions
        elif from_unit == "kg" and to_unit == "pounds":
            result = value * 2.205
            return f"{value} kg = {result:.2f} pounds"

        elif from_unit == "pounds" and to_unit == "kg":
            result = value / 2.205
            return f"{value} pounds = {result:.2f} kg"

        else:
            return f"Error: Unsupported conversion from '{from_unit}' to '{to_unit}'."

    except ValueError:
        return "Error: The value must be a valid number."

# Tests
print(unit_converter.invoke("5 km to miles"))
print(unit_converter.invoke("10 kg to pounds"))
print(unit_converter.invoke("3.1 miles to km"))
print(unit_converter.invoke("22 pounds to kg"))

5.0 km = 3.10 miles
10.0 kg = 22.05 pounds
3.1 miles = 4.99 km
22.0 pounds = 9.98 kg


### C3. Assemble + Stress Test

1. Build an agent with `grade_calculator`, `unit_converter`, and the class `calculator`.
2. Ask it this tricky question and observe:

> "I scored 78 out of 100 and my school is 3 km away. What is my grade and how far is that in miles?"

3. Then ask a question where the agent must **calculate first, then grade**:

> "I got 45, 60, and 75 in three tests (each out of 100). What is my average, and what grade is that?"

Does it chain the tools correctly? Note the order of Actions.

In [ ]:
# TODO: build agent + executor and run both questions

In [ ]:
from langchain.tools import tool

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression.
    Input examples:
    2+3
    (45+60+75)/3
    2**10
    """

    # Remove surrounding quotes if present
    expression = expression.strip().strip("'\"")

    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

In [ ]:
# Build the agent
prompt = PromptTemplate.from_template("""
Answer the following questions as best you can.

You have access to the following tools:

{tools}

Use the following format:

Question: the input question
Thought: you should always think about what to do
Action: one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original question

Begin!

Question: {input}
Thought:{agent_scratchpad}
""")







agent = create_react_agent(
    llm=llm,
    tools=[calculator, grade_calculator, unit_converter],
    prompt=prompt          # Use the ReAct prompt from class
)

# Build the executor
executor = AgentExecutor(
    agent=agent,
    tools=[calculator, grade_calculator, unit_converter],
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

# -----------------------
# Test 1
# -----------------------
response1 = executor.invoke({
    "input": "I scored 78 out of 100 and my school is 3 km away. What is my grade and how far is that in miles?"
})

print(response1["output"])

# -----------------------
# Test 2
# -----------------------
response2 = executor.invoke({
    "input": "I got 45, 60, and 75 in three tests (each out of 100). What is my average, and what grade is that?"
})

print(response2["output"])



> Entering new AgentExecutor chain...
Thought: To solve this problem, I need to calculate the grade for the given score and convert the distance from kilometers to miles. First, I should calculate the grade.

Action: grade_calculator
Action Input: 78Grade: BNow that I have the grade, I need to convert the distance from kilometers to miles.

Action: unit_converter
Action Input: 3 km to miles3.0 km = 1.86 milesI now know the final answer

Final Answer: Your grade is B and the distance to your school is 1.86 miles.

> Finished chain.
Your grade is B and the distance to your school is 1.86 miles.


> Entering new AgentExecutor chain...
Thought: To find the average of the three test scores, I need to add them up and divide by the number of tests. I can use the calculator to evaluate this mathematical expression.

Action: calculator
Action Input: (45+60+75)/360.0Thought: Now that I have the average score, I need to calculate the grade for this average score. I can use the grade_calculator 

**Observations (order of tools used, any mistakes):**

...

---
# Part D — Mini Project: Personal Study Assistant 🎓 (Graded)

Build a complete **Study Assistant Agent with memory** that has these 3 tools:

1. `syllabus_lookup(subject)` — dictionary of at least 4 subjects → topic lists (e.g. `"python": "variables, loops, functions, OOP"`).
2. `study_time(topic)` — dictionary of topics → estimated hours (e.g. `"loops": "3 hours"`).
3. `grade_calculator` — reuse from C1.

Then run this **4-turn conversation** with the SAME executor (memory must work):

1. `"Hi, I am <your name> and I want to learn Python."`
2. `"What topics are in the Python syllabus?"`
3. `"How many hours do I need for loops?"`
4. `"What was my name and which subject did I want to learn?"` ← memory test!

**Submit:** the full verbose output of turn 4.

In [ ]:
# Mini project workspace

# TODO: Tool 1 — syllabus_lookup
# TODO: Tool 2 — study_time
# TODO: reuse grade_calculator
# TODO: memory prompt (must include {chat_history})
# TODO: ConversationBufferMemory
# TODO: agent + executor with memory
# TODO: run the 4 turns

In [ ]:
@tool
def syllabus_lookup(subject: str) -> str:
    """
    Look up the syllabus topics for a subject.
    Input should be the subject name.
    """
    syllabus = {
        "python": "Variables, Data Types, Loops, Functions, OOP",
        "java": "Variables, Classes, Objects, Inheritance, Exception Handling",
        "dbms": "ER Model, SQL, Joins, Transactions, Normalization",
        "ai": "Search Algorithms, Machine Learning, Neural Networks, NLP"
    }

    subject = subject.lower().strip().strip("'\"")

    return syllabus.get(subject, "Subject not found.")

@tool
def study_time(topic: str) -> str:
    """
    Return the estimated study time for a topic.
    """
    times = {
        "variables": "2 hours",
        "data types": "2 hours",
        "loops": "3 hours",
        "functions": "4 hours",
        "oop": "6 hours",
        "sql": "5 hours",
        "joins": "4 hours",
        "normalization": "3 hours",
        "neural networks": "8 hours",
        "nlp": "10 hours"
    }

    topic = topic.lower().strip().strip("'\"")

    return times.get(topic, "Study time not available.")

In [ ]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=False
)

In [ ]:
prompt = PromptTemplate.from_template("""
You are a helpful Study Assistant.

Previous conversation:
{chat_history}

You have access to these tools:

{tools}



Only use a tool if it is necessary.

If the answer is already available from the conversation history,
DO NOT use any tool.
Instead respond directly with:

Thought: I know the answer from memory.
Final Answer: ...


Use the following format:

Question: the input question
Thought: think carefully
Action: one of [{tool_names}]
Action Input: the input
Observation: the result
... (repeat as needed)
Thought: I now know the answer
Final Answer: the answer

Question: {input}
Thought:{agent_scratchpad}
""")

In [ ]:
agent = create_react_agent(
    llm=llm,
    tools=[
        syllabus_lookup,
        study_time,
        grade_calculator
    ],
    prompt=prompt
)

executor = AgentExecutor(
    agent=agent,
    tools=[
        syllabus_lookup,
        study_time,
        grade_calculator
    ],
    memory=memory,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

In [ ]:
executor.invoke(
    {"input":"Hi, I am Ron and I want to learn Python."}
)

executor.invoke(
    {"input":"What topics are in the Python syllabus?"}
)

executor.invoke(
    {"input":"How many hours do I need for loops?"}
)

result = executor.invoke(
    {"input":"What was my name and which subject did I want to learn?"}
)

print(result["output"])



> Entering new AgentExecutor chain...
Thought: To help Ron learn Python, I should first look up the syllabus topics for the subject "Python" to understand what needs to be covered.
Action: syllabus_lookup
Action Input: PythonVariables, Data Types, Loops, Functions, OOPThought: Now that I have the syllabus topics for Python, I can see that it covers Variables, Data Types, Loops, Functions, and OOP. To provide a more detailed plan for Ron, I should estimate the study time required for each of these topics.
Action: study_time
Action Input: Variables2 hoursThought: I have estimated the study time for the "Variables" topic, which is 2 hours. Next, I should estimate the study time for the remaining topics: "Data Types", "Loops", "Functions", and "OOP".

Action: study_time
Action Input: Data Types2 hoursThought: I have estimated the study time for the "Variables" and "Data Types" topics, which are both 2 hours. Next, I should estimate the study time for the remaining topics: "Loops", "Funct

---
## ⭐ Bonus (Optional): One Tool to Rule Them All?

Instead of 3 separate tools, could you build ONE tool `study_helper(query)` that does everything with if/else inside?

Try it, then answer: **Why is this a BAD design for agents?** (Hint: think about what the docstring would have to say, and how the agent decides which tool to pick.)

**Answer:**


Using one study_helper tool is a bad design because it combines multiple unrelated tasks into a single function. Its docstring becomes long and ambiguous, making it harder for the agent to understand when to use it. The agent can no longer choose between specialized tools, reducing the accuracy of tool selection. It also makes the code harder to maintain, test, and extend. Designing small, single-purpose tools with clear descriptions is a better practice for building reliable AI agents.

In [ ]:
from langchain.tools import tool
import re

@tool
def study_helper(query: str) -> str:
    """
    A study assistant that can:
    - Look up syllabus topics for a subject.
    - Estimate study time for a topic.
    - Calculate grades from marks.
    """

    query = query.lower().strip()

    syllabus = {
        "python": "Variables, Data Types, Loops, Functions, OOP",
        "java": "Variables, Classes, Objects, Inheritance, Exception Handling",
        "dbms": "ER Model, SQL, Joins, Transactions, Normalization",
        "ai": "Search, Machine Learning, Neural Networks, NLP"
    }

    study_times = {
        "variables": "2 hours",
        "loops": "3 hours",
        "functions": "4 hours",
        "oop": "6 hours",
        "sql": "5 hours",
        "joins": "4 hours",
        "normalization": "3 hours",
        "machine learning": "10 hours",
        "neural networks": "8 hours",
        "nlp": "10 hours"
    }

    # Grade calculation
    if "grade" in query or "marks" in query:
        nums = re.findall(r"\d+", query)
        if not nums:
            return "No marks found."

        marks = int(nums[0])

        if not 0 <= marks <= 100:
            return "Marks must be between 0 and 100."

        if marks >= 90:
            return "Grade: A"
        elif marks >= 75:
            return "Grade: B"
        elif marks >= 60:
            return "Grade: C"
        elif marks >= 40:
            return "Grade: D"
        else:
            return "Grade: Fail"

    # Syllabus lookup
    for subject in syllabus:
        if subject in query:
            return f"{subject.title()} syllabus: {syllabus[subject]}"

    # Study time lookup
    for topic in study_times:
        if topic in query:
            return f"{topic.title()} requires approximately {study_times[topic]}."

    return "Sorry, I couldn't understand your request."

In [ ]:
print(study_helper.invoke("Show me the Python syllabus"))
print(study_helper.invoke("Show me the Java syllabus"))
print(study_helper.invoke("How much time for loops?"))
print(study_helper.invoke("How much time for SQL?"))
print(study_helper.invoke("My marks are 92"))

Python syllabus: Variables, Data Types, Loops, Functions, OOP
Java syllabus: Variables, Classes, Objects, Inheritance, Exception Handling
Loops requires approximately 3 hours.
Sql requires approximately 5 hours.
Grade: A


---
## ✅ Submission Checklist

- [ ] Part A: All 3 predictions/answers written
- [ ] Part B: All 5 bugs found, fixed, and explained
- [ ] Part C: Both tools pass their test cells; stress test observations written
- [ ] Part D: 4-turn memory conversation works, turn-4 output pasted
- [ ] Bonus (optional)

**Tip:** `Runtime → Restart and run all` before submitting.